In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("../EDA/modeling_dataset_baseline.csv")
df['date'] = pd.to_datetime(df['date'])

In [7]:
### Feature construction

# Construct house age from date and year built
df['house_age'] = df['date'].dt.year - df['yr_built']
df['house_age'] = df['house_age'].clip(lower=0)
# Construct an indicator variable for if the house has ever been renovated.
df['renovated_bool'] = [1 if x > 0 else 0 for x in df['yr_renovated']]

# Create a few logged variables for visualization and analysis
df['log_price'] = np.log(df['price']).round(4)
df['log_sqft_living'] = np.log(df['sqft_living']).round(4)
df['log_sqft_above'] = np.log(df['sqft_above']).round(4)
df['log_sqft_basement'] = np.log1p(df['sqft_basement']).round(4)
df['log_sqft_lot'] = np.log(df['sqft_lot']).round(4)
df['log_sqft_lot15'] = np.log(df['sqft_lot15']).round(4)

# Living to lot ratio
df['living_lot_ratio'] = (df['sqft_living'] / df['sqft_lot']).round(3)

# Month of sale (that may influence the price)
df['sale_month'] = df['date'].dt.month

# Grade /condition groups - lets visualize these in the next step to see if these are meaningful or if we can improve them
df['grade_level'] = pd.cut(df['grade'], bins=[0, 6, 9, 13], labels=['Low', 'Mid', 'High'])
df['condition_level'] = pd.cut(df['condition'], bins = [0, 2, 3, 5], 
                                          labels = ['Poor', 'Average', 'Good'])

engineered_full = df.copy()

In [8]:
# Check for extreme outliers 3 times the IQR above or below the third or first quartiles.
cols_to_filter = ['price', 'bedrooms', 'sqft_living', 'grade', 'sqft_above', 'sqft_basement']
# Make not of indices to drop
to_drop_indices = []

for col in cols_to_filter:
    # Calculate the outliers
    q1 = engineered_full[col].quantile(0.25)
    q3 = engineered_full[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 3.0 * iqr
    upper_bound = q3 + 3.0 * iqr
    # Grab the indices to drop
    extreme_outliers = engineered_full[(engineered_full[col] > upper_bound) | (engineered_full[col] < lower_bound)].index
    to_drop_indices.extend(extreme_outliers)
# Drop the outlier indices
engineered_trimmed = engineered_full.drop(index=list(set(to_drop_indices)))

In [9]:
# Columns to drop - drop sqft_living in favor of above and basement.
cols_to_drop = ['id', 'date', 'yr_built', 'yr_renovated',
                'zipcode', 'lat', 'long']

engineered_full = engineered_full.drop(columns=cols_to_drop)
engineered_trimmed = engineered_trimmed.drop(columns=cols_to_drop)

In [10]:
# Export both sets
engineered_full.to_csv("../Model/modeling_engineered_full.csv", index=False)
engineered_trimmed.to_csv("../Model/modeling_engineered_trimmed.csv", index=False)

In [11]:
# Validation Checks

print(engineered_full.isna().sum().sort_values(ascending=False).head(10))

print(engineered_full.shape)
print(engineered_trimmed.shape)

print(engineered_full.columns.equals(engineered_trimmed.columns))

def check_numeric_issues(df, name):
    print(f"\nChecking {name}")
    print("NaNs:", df.isna().sum().sum())
    print("Inf:", np.isinf(df.select_dtypes(include=[np.number])).sum().sum())

check_numeric_issues(engineered_full, "engineered_full")
check_numeric_issues(engineered_trimmed, "engineered_trimmed")

price                0
bedrooms             0
grade_level          0
sale_month           0
living_lot_ratio     0
log_sqft_lot15       0
log_sqft_lot         0
log_sqft_basement    0
log_sqft_above       0
log_sqft_living      0
dtype: int64
(21613, 26)
(21096, 26)
True

Checking engineered_full
NaNs: 0
Inf: 0

Checking engineered_trimmed
NaNs: 0
Inf: 0
